<a href="https://colab.research.google.com/github/kavyasudha12/Gen-AI-Training-Task/blob/main/RAG_CHATBOT.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
# INSTALL
!pip -q install openai pypdf faiss-cpu gradio numpy


# SET API KEY

from google.colab import userdata
import os

os.environ["OPENAI_API_KEY"] = userdata.get("MY_OPENAI_KEY")



# IMPORTS
import re
import numpy as np
import faiss
import gradio as gr

from openai import OpenAI
from pypdf import PdfReader

client = OpenAI()


# DOCUMENT FUNCTIONS
def extract_text_from_pdf(path):
    reader = PdfReader(path)
    text = []
    for page in reader.pages:
        t = page.extract_text()
        if t:
            text.append(t)
    return "\n".join(text)

def extract_text_from_txt(path):
    with open(path, "r", encoding="utf-8", errors="ignore") as f:
        return f.read()

def load_document(path):
    if path.endswith(".pdf"):
        return extract_text_from_pdf(path)
    elif path.endswith(".txt"):
        return extract_text_from_txt(path)
    else:
        raise ValueError("Only PDF/TXT supported")

def clean_text(text):
    return re.sub(r"\s+", " ", text).strip()

def chunk_text(text, size=900, overlap=150):
    text = clean_text(text)
    chunks = []
    step = size - overlap
    i = 0
    while i < len(text):
        chunk = text[i:i+size]
        if chunk.strip():
            chunks.append(chunk)
        i += step
    return chunks


# OPENAI HELPERS
def get_embedding(text):
    res = client.embeddings.create(
        model="text-embedding-3-small",
        input=text if text.strip() else "empty"
    )
    return np.array(res.data[0].embedding, dtype="float32")

def ask_llm(prompt):
    res = client.responses.create(
        model="gpt-4.1-mini",
        input=prompt
    )
    return res.output_text.strip()

def normalize_question(q):
    prompt = f"Fix spelling and rewrite clearly: {q}"
    return ask_llm(prompt)

def answer_from_context(q, context):
    prompt = f"""
Answer briefly using ONLY the context.
If not found, return NOT_FOUND.

Context:
{context}

Question:
{q}
"""
    return ask_llm(prompt)

def fallback_answer(q):
    prompt = f"""
Answer this using general knowledge (not from document).
Keep it short and natural.

Question:
{q}
"""
    return ask_llm(prompt)


# RAG ENGINE
class RAG:
    def __init__(self):
        self.chunks = []
        self.index = None
        self.loaded = False

    def build(self, text):
        self.chunks = chunk_text(text)

        embeddings = np.array([get_embedding(c) for c in self.chunks], dtype="float32")
        self.index = faiss.IndexFlatL2(embeddings.shape[1])
        self.index.add(embeddings)

        self.loaded = True

    def retrieve(self, query, k=4):
        q = np.array([get_embedding(query)], dtype="float32")
        D, I = self.index.search(q, k)

        results = []
        for idx, dist in zip(I[0], D[0]):
            if idx == -1:
                continue
            results.append(self.chunks[int(idx)])
        return results

rag = RAG()


# CHAT LOGIC
def make_reply(msg):
    if not rag.loaded:
        return "Please upload a file first."

    q = normalize_question(msg)
    chunks = rag.retrieve(q)

    context = "\n\n".join(chunks)
    ans = answer_from_context(q, context)

    if ans.strip() == "NOT_FOUND":
        return fallback_answer(q)

    return ans

# FILE PROCESSING
def process_file(file):
    if file is None:
        return "Upload a file first"

    try:
        text = load_document(file.name)
        if len(text) < 50:
            return "File is empty"

        rag.build(text)
        return "File processed successfully"
    except Exception as e:
        return str(e)


# CHAT FUNCTION
def respond(msg, history):
    answer = make_reply(msg)
    history.append({"role": "user", "content": msg})
    history.append({"role": "assistant", "content": answer})
    return history, ""


# UI
css = """
body {background:#f7f7f8}
footer {display:none !important}
"""

with gr.Blocks(css=css) as demo:

    gr.Markdown("## Mini RAG Chatbot")

    with gr.Row():
        file = gr.File(file_types=[".pdf", ".txt"])
        btn = gr.Button("Process")

    status = gr.Markdown("Upload a document")

    chat = gr.Chatbot(type="messages", height=500)

    msg = gr.Textbox(
        placeholder="Ask something...",
        show_label=False
    )

    clear = gr.Button("Clear")

    btn.click(process_file, file, status)
    msg.submit(respond, [msg, chat], [chat, msg])
    clear.click(lambda: [], None, chat)


# LAUNCH (FIXED)
gr.close_all()

demo.launch(
    share=True,
    inline=False,
    debug=False,
    quiet=True,
    show_error=False
)



/tmp/ipykernel_10701/4047213232.py:183: DeprecationWarning: The 'css' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'css' to Blocks.launch() instead.
  with gr.Blocks(css=css) as demo:
/tmp/ipykernel_10701/4047213232.py:193: DeprecationWarning: The default value of 'allow_tags' in gr.Chatbot will be changed from False to True in Gradio 6.0. You will need to explicitly set allow_tags=False if you want to disable tags in your chatbot.
  chat = gr.Chatbot(type="messages", height=500)


Closing server running on port: 7860
* Running on public URL: https://14f2c6a159e9bf13b1.gradio.live
